## Settings

In [1]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [2]:
## libraries
import sys
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.resampling import (
    logo_cross_valid,
    kfold_cross_valid,
    compile_cross_valid,
    results_cross_valid
)
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z, 
    TARGET
)

## Initalization

In [3]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## setup
data = load_processed_data()
models = load_estimators()

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

Original Data: 32 features, 25 observations


## Training

In [4]:
## leave-one-group-out cross validation (domain)
if "results_dict_domain" not in globals():
    results_dict_domain = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "domain",  ## fold on domain
            n_repeats = 30,
            random_state = 42
        )
        frontier["model"] = name
        results_dict_domain[name] = frontier


Training linear_quantile...
Training linear_convex...
Training linear_laws...
Training forest_quantile...
Training boosted_quantile...
Training xgboost_quantile...
Training neural_quantile...
Training neural_expectile...
Training neural_convex...


In [5]:
## leave-one-group-out cross validation (discipline)
if "results_dict_disc" not in globals():
    results_dict_disc = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "discipline",  ## fold on discipline
            n_repeats = 30,
            random_state = 42
        )
        frontier["model"] = name
        results_dict_disc[name] = frontier


Training linear_quantile...
Training linear_convex...
Training linear_laws...
Training forest_quantile...
Training boosted_quantile...
Training xgboost_quantile...
Training neural_quantile...
Training neural_expectile...
Training neural_convex...


In [6]:
## 5-fold cross validation
if "results_dict_5fold" not in globals():
    results_dict_5fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 5,  ## fold on random 80-20 split
            n_repeats = 30,
            random_state = 42
        )
        frontier["model"] = name
        results_dict_5fold[name] = frontier


Training linear_quantile...
Training linear_convex...
Training linear_laws...
Training forest_quantile...
Training boosted_quantile...
Training xgboost_quantile...
Training neural_quantile...
Training neural_expectile...
Training neural_convex...


In [7]:
## 10-fold cross validation
if "results_dict_10fold" not in globals():
    results_dict_10fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, _ = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 10,  ## fold on random 90-10 split
            n_repeats = 30,
            random_state = 42
        )
        frontier["model"] = name
        results_dict_10fold[name] = frontier


Training linear_quantile...
Training linear_convex...
Training linear_laws...
Training forest_quantile...
Training boosted_quantile...
Training xgboost_quantile...
Training neural_quantile...
Training neural_expectile...
Training neural_convex...


## Post-Processing

In [8]:
## convert all results to dataframes
results_data_domain = compile_cross_valid(results = results_dict_domain)
results_data_disc = compile_cross_valid(results = results_dict_disc)
results_data_5fold = compile_cross_valid(results = results_dict_5fold)
results_data_10fold = compile_cross_valid(results = results_dict_10fold)

## Cross-Validation Tests
Four resampling regimes evaluate generalization under progressively easier conditions. Transfer regimes hold out entire domains or disciplines. Baseline regimes use standard random partitions.

- **Domain (LOGO)**: Leave-one-group-out by domain (5 groups). For each model, frontier metrics are averaged over held-out domains. Repeats vary learner initialization. The held-out domain splits stay fixed.
- **Discipline (LOGO)**: Leave-one-group-out by discipline (25 groups). For each model, frontier metrics are averaged over held-out disciplines. Repeats vary learner initialization. The held-out discipline splits stay fixed.
- **Random (5-Fold)**: Repeated shuffled 80–20 splits. For each model, frontier metrics are averaged over repeated random folds.
- **Random (10-Fold)**: Repeated shuffled 90–10 splits. For each model, frontier metrics are averaged over repeated random folds.

### Reporting Convention
The table unit is one model-level summary per regime. Within each model, metrics are averaged over groups or folds and repeats. Across models, the table reports the median of those model-level means. EI is reported as median [Q1–Q3]. VR is violation rate. MV and MS are reported on the log-target scale. EI normalizes MV and MS internally, but the MV and MS columns do not. Results are equally weighted across groups, folds, repeats, and models. They are not observation-weighted.

In [11]:
## compile transfer bounds and baseline benchmarks
results = results_cross_valid(
    {"Domain (LOGO)": results_data_domain, "Discipline (LOGO)": results_data_disc},
    {"Random (5-Fold)": results_data_5fold, "Random (10-Fold)": results_data_10fold},
    indicies = ("Evaluation", "Hold-Out"),
    keys = ("Transfer", "Standard"),
    n_repeats = N_REPEATS,
    random_state = RANDOM_STATE,
    decimals = 3
)

display(results)

Cross-Validation: 9 models, 30 repeats (seeds 42–71)
Across-model aggregation: median of model-level means
Resampling: LOGO splits are fixed across repeats; random k-fold splits are reshuffled across repeats
Weighting: groups, folds, repeats, and models are equally weighted; results are not observation-weighted


Transfer IQR spreads are narrow relative to their median EI. Cross-domain generalization is stable across learning paradigms. Baseline regimes confirm the expected upper bound under random partitioning. EI generalizes beyond the domains on which it was trained.